In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression, ElasticNet
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
)
from sklearn.model_selection import StratifiedGroupKFold

In [2]:
with open('/Users/samuelwright/Documents/CSE 283/Project/trained_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

/Users/samuelwright/miniconda3/envs/king_base/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/samuelwright/miniconda3/envs/king_base/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.3.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/samuelwright/miniconda3/envs/king_base/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpick

In [6]:
import sys
sys.path.append('/Users/samuelwright/Documents/CSE 283/Project/ad_classifier_pipeline.py')
from ad_classifier_pipeline import load_counts, load_metadata, load_gwas_genes, align_samples, filter_low_counts, normalize, build_ensembl_to_symbol_map, gwas_filter_expression, log2fc_filter_expression, encode_covariates, build_feature_matrix

In [ ]:
counts_path = '/Users/samuelwright/Documents/CSE 283/Project/toden_counts.txt'
meta_path = '/Users/samuelwright/Documents/CSE 283/Project/toden_metadata.xlsx'
gwas_csv = '/Users/samuelwright/Documents/CSE 283/Project/AD_GWAS_hits_converted.csv'
norm_method = 'cpm_log2'
min_count: int = 10,
min_samples_frac: float = 0.10,

counts = load_counts(counts_path)
meta = load_metadata(meta_path)
gwas_genes = load_gwas_genes(gwas_csv)
counts, meta = align_samples(counts, meta)
counts_filt = filter_low_counts(counts, min_count, min_samples_frac)
norm_expr = normalize(counts_filt, meta, method=norm_method)
ensembl_map = build_ensembl_to_symbol_map(norm_expr.index)
gwas_expr   = gwas_filter_expression(norm_expr, gwas_genes, ensembl_map)
log2fc_genes = log2fc_filter_expression(counts, meta,n_genes=100)
combined_expr = pd.concat([gwas_expr,log2fc_genes]).drop_duplicates()
covariates  = encode_covariates(meta)
X = build_feature_matrix(gwas_expr, covariates)
common_samples = X.index.intersection(meta.index)
X = X.loc[common_samples]
meta = meta.loc[common_samples]
y = (meta["donor_group"] == "AD").astype(int).rename("label")
groups = meta["donor_id_alias"] 